# Загрузка и предобработка данных

In [1]:
# Импортируем библиотеки
import pandas as pd
import json
import numpy as np

In [2]:
# Загружаем исходный CSV
df = pd.read_csv(r"C:\Users\anton\Downloads\ДТП\ДТП\_output\dtp_combined.csv", low_memory=False, parse_dates=['datetime'])

# Создаём таблицу ДТП (accidents)
accidents = df.drop(columns=['vehicles', 'participants']).copy()

# Создаём таблицу транспортных средств (vehicles)
vehicles_data = []
for _, row in df.iterrows():
    accident_id = row['id']
    try:
        vehicles_list = json.loads(row['vehicles'].replace("'", '"'))
        for v in vehicles_list:
            v['accident_id'] = accident_id
            vehicles_data.append(v)
    except Exception:
        pass  # Пропускаем пустые или некорректные строки

vehicles = pd.DataFrame(vehicles_data)

# Создаём таблицу участников (participants)
participants_data = []
for _, row in df.iterrows():
    accident_id = row['id']
    try:
        participants_list = json.loads(row['participants'].replace("'", '"'))
        for p in participants_list:
            p['accident_id'] = accident_id
            participants_data.append(p)
    except Exception:
        pass

participants = pd.DataFrame(participants_data)

# Сохраняем таблицы в отдельные CSV
accidents.to_csv('accidents.csv', index=False)
vehicles.to_csv('vehicles.csv', index=False)
participants.to_csv('participants.csv', index=False)

print(f"Таблицы сохранены: {len(accidents)} ДТП, {len(vehicles)} ТС, {len(participants)} участников.")

Таблицы сохранены: 1465882 ДТП, 2325731 ТС, 455224 участников.


In [3]:
# Загружаем таблицы 
accidents = pd.read_csv('accidents.csv', low_memory=False)
vehicles = pd.read_csv('vehicles.csv')
participants = pd.read_csv('participants.csv')

In [4]:
# Очистка accidents от дубликатов и лишних данных
accidents = (accidents
             .drop(columns=['source_file', 'scheme'], errors='ignore')
             .drop_duplicates()
             .reset_index(drop=True))
accidents.head()

,id,light,point,region,address,category,datetime,severity,dead_count,injured_count,parent_region,participants_count,geometry,nearby
0,189548,Светлое время суток,"{ ""lat"": 49.789873999999998, ""long"": 129.84691...",Бурейский район,"Обход п. Бурея, 5 км",Наезд на пешехода,2019-04-05 14:10:00,Легкий,0,1,Амурская область,2,POINT (129.846919 49.789874),NaN
1,188700,"В темное время суток, освещение отсутствует","{ ""lat"": 49.755600000000001, ""long"": 129.29220...",Бурейский район,"Благовещенск – Гомелевка, 148 км",Столкновение,2022-11-16 22:50:00,С погибшими,1,1,Амурская область,3,POINT (129.2922 49.7556),NaN
2,188702,"В темное время суток, освещение не включено","{ ""lat"": 49.787187000000003, ""long"": 129.81648...",Бурейский район,"Обход п. Бурея, 2 км",Съезд с дороги,2019-12-27 23:50:00,Легкий,0,1,Амурская область,1,POINT (129.816481 49.787187),NaN
3,188704,"В темное время суток, освещение отсутствует","{ ""lat"": 49.135599999999997, ""long"": 129.1181 }",Бурейский район,"пгт Бурея, ул Райчихинская, 1",Наезд на пешехода,2016-12-30 19:10:00,С погибшими,1,0,Амурская область,2,POINT (129.1181 49.1356),NaN
4,188706,Светлое время суток,"{ ""lat"": 49.119199999999999, ""long"": 129.1353 }",Бурейский район,"пгт Новобурейский, ул Лесная, 22",Наезд на пешехода,2017-12-30 08:45:00,Легкий,0,1,Амурская область,2,POINT (129.1353 49.1192),NaN


Мы решили удалить столбец `scheme`, так как смысловой нагрузки он не несет.

In [5]:
# Очистка vehicles от дубликатов и лишних данных
vehicles = (vehicles
            .drop(columns=['participants'], errors='ignore')
            .drop_duplicates()
            .reset_index(drop=True))
vehicles['year'] = pd.to_numeric(vehicles['year'], errors='coerce').astype('Int64')

# Установим корректность связей между таблицами
vehicles = vehicles[vehicles['accident_id'].isin(accidents['id'])]
vehicles.head()

,year,brand,color,model,category,accident_id
0,2005,TOYOTA,Серый,Paseo,"С-класс (малый средний, компактный) до 4,3 м",189548
1,1993,TOYOTA,Белый,Corona,Прочие легковые автомобили,188700
2,1988,ISUZU,Синий,Прочие модели Isuzu,Фургоны,188700
3,2008,TOYOTA,Белый,Prius,"D-класс (средний) до 4,6 м",188702
4,1989,TOYOTA,Белый,Ipsum,"D-класс (средний) до 4,6 м",188704


In [6]:
# Очистка participants от дубликатов
participants.drop_duplicates().reset_index(drop=True)

# Установим корректность связей между таблицами
participants = participants[participants['accident_id'].isin(accidents['id'])]
participants.head()

,role,gender,violations,health_status,accident_id
0,Пешеход,Мужской,['Нахождение на проезжей части без цели её пер...,"Раненый, находящийся (находившийся) на амбула...",189548
1,Пешеход,Мужской,['Нахождение на проезжей части без цели её пер...,Скончался в течение 1 суток,188704
2,Пешеход,Женский,[],"Раненый, находящийся (находившийся) на амбула...",188706
3,Пешеход,Женский,['Нахождение на проезжей части без цели её пер...,"Раненый, находящийся (находившийся) на стацион...",188709
4,Пешеход,Женский,['Переход через проезжую часть вне пешеходного...,"Раненый, находящийся (находившийся) на амбулат...",188711


In [7]:
accidents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1465882 entries, 0 to 1465881
Data columns (total 14 columns):
 #   Column              Non-Null Count    Dtype 
---  ------              --------------    ----- 
 0   id                  1465882 non-null  int64 
 1   light               1465882 non-null  object
 2   point               1465882 non-null  object
 3   region              1465882 non-null  object
 4   address             1389322 non-null  object
 5   category            1465882 non-null  object
 6   datetime            1465882 non-null  object
 7   severity            1465882 non-null  object
 8   dead_count          1465882 non-null  int64 
 9   injured_count       1465882 non-null  int64 
 10  parent_region       1465882 non-null  object
 11  participants_count  1465882 non-null  int64 
 12  geometry            1456071 non-null  object
 13  nearby              500849 non-null   object
dtypes: int64(4), object(10)
memory usage: 156.6+ MB


In [8]:
vehicles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2313987 entries, 0 to 2313986
Data columns (total 6 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   year         Int64 
 1   brand        object
 2   color        object
 3   model        object
 4   category     object
 5   accident_id  int64 
dtypes: Int64(1), int64(1), object(4)
memory usage: 108.1+ MB


In [9]:
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 455224 entries, 0 to 455223
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   role           455224 non-null  object
 1   gender         447672 non-null  object
 2   violations     455224 non-null  object
 3   health_status  455194 non-null  object
 4   accident_id    455224 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 17.4+ MB


In [10]:
# Отчёт по пропускам
def missing_report(df, name):
    print(f"\nПропуски в {name}:")
    print(df.isna().sum()[df.isna().sum() > 0])

missing_report(accidents, "ACCIDENTS")
missing_report(vehicles, "VEHICLES")
missing_report(participants, "PARTICIPANTS")


Пропуски в ACCIDENTS:
address      76560
geometry      9811
nearby      965033
dtype: int64

Пропуски в VEHICLES:
year     88740
brand    79490
color    50844
model    79510
dtype: int64

Пропуски в PARTICIPANTS:
gender           7552
health_status      30
dtype: int64


В таблице `accidents` мы решили оставить пропуски, чтобы корректно проводить анализ географии. В колонке `nearby` пропусков слишком много, их удаление может повести за собой некорректный анализ.

In [11]:
# Обработка пропусков в таблицах
vehicles['brand'] = vehicles['brand'].fillna('UNKNOWN')
vehicles['color'] = vehicles['color'].fillna('UNKNOWN')
vehicles['model'] = vehicles['model'].fillna('UNKNOWN')

participants['gender'] = participants['gender'].fillna('Не указан')
participants['health_status'] = participants['health_status'].fillna('Не указан')

В таблице `vehicles` мы заполнили пропуски неизвестным значением в колонках `brand`, `color`, `model` для анализа частоты ДТП, а в колонке `year` решили оставить пропуски, так как заполнение пропусков средним годом может привести к некорректным данным.

В таблице `participants` мы заменили пропуски на неизвестное значение, так как такие данные возможно просто не были указаны.

In [12]:
# Проверка аномалий в accidents
accidents_anomalies = accidents[
    (accidents['dead_count'] < 0) |
    (accidents['injured_count'] < 0) |
    (accidents['participants_count'] < 0) |
    (accidents['dead_count'] > accidents['participants_count'])
]
print(f"Аномалии в accidents: {len(accidents_anomalies)} строк")

Аномалии в accidents: 0 строк


In [13]:
print(accidents['datetime'].min())
print(accidents['datetime'].max())

2015-01-01 00:01:00
2025-06-30 23:10:00


In [14]:
# Проверка аномалий в vehicles
vehicles_anomalies = vehicles[
    (vehicles['year'] > 2025)
]
print(f"Аномалии в vehicles: {len(vehicles_anomalies)} строк")

Аномалии в vehicles: 0 строк


In [15]:
vehicles['year'].unique()

<IntegerArray>
[2005, 1993, 1988, 2008, 1989, 2003, 2009, 2017, 1994, 1987, 2000, 1999, 2013,
 2001, 2024, 2004, 1998, 2002, 1984, 2011, 2021, 2010, 2014, 1986, 1992, 1983,
 2023, 2016, 1991, 2007, 1995, 2019, 1996, 2018, 1997, 2022, 2012, 1985, <NA>,
 2006, 1982, 1980, 2015, 2020, 1990, 1974, 1975, 1981, 1979, 1978, 1977, 1969,
 1972, 1967, 1968,    1, 2025, 1959, 1976, 1963, 1970, 1973, 1962, 1900, 1971,
 1964, 1960, 1958, 1965, 1961, 1955, 1910, 1966, 1952, 1950, 1949, 1956, 1911,
 1953, 1909, 1919, 1954, 1957, 1948, 1928, 1920, 1941, 1937, 1918, 1936, 1907,
 1942, 1946, 1934, 1901, 1923]
Length: 96, dtype: Int64

Видим аномалии в столбце `year`. Машина раньше 1970 года выпуска маловероятно может быть на ходу с 2015 года, поэтому отфильтруем от 1970 по 2025 год.

In [16]:
vehicles = vehicles[(vehicles['year'] >= 1970) & (vehicles['year'] <= 2025)]

In [17]:
# Добавим колонку с возрастом авто
vehicles['vehicle_age'] = 2025 - vehicles['year']
print(vehicles['vehicle_age'].describe())

count    2224003.0
mean     16.532838
std       7.976804
min            0.0
25%           11.0
50%           15.0
75%           21.0
max           55.0
Name: vehicle_age, dtype: Float64


В среднем автомобилю 15-16 лет, минимальное значение - 0, значит совсем новое авто попало в ДТП, максимальное - 55 лет.

In [18]:
# Проверка аномалий в participants
participants_anomalies = participants['gender'].unique()
participants_anomalies

array(['Мужской', 'Женский', 'Не указан'], dtype=object)

In [19]:
# Конвертируем очищенные данные в csv-файлы
accidents.to_csv('accidents_clean.csv', index=False)
vehicles.to_csv('vehicles_clean.csv', index=False)
participants.to_csv('participants_clean.csv', index=False)

Вывод:
- Загрузили исходный csv-файл со всеми данными по ДТП, затем поделили его на три таблицы: `accidents` - ДТП, `vehicles` - ТС, `participants` - участники ДТП.
- Удалили столбец `scheme`, так как смысловой нагрузки он не несет.
- Типы данных корректные.
- Дубликатов нет.
- В таблице `accidents` мы решили оставить пропуски, чтобы корректно проводить анализ географии. В колонке `nearby` пропусков слишком много, их удаление может повести за собой некорректный анализ.
- В таблице `vehicles` мы заполнили пропуски неизвестным значением в колонках `brand`, `color`, `model` для анализа частоты ДТП, а в колонке `year` решили оставить пропуски, так как заполнение пропусков средним годом может привести к некорректным данным.
- В таблице `participants` мы заменили пропуски на неизвестное значение, так как такие данные возможно просто не были указаны.
аномалии мы увидели только в столбце `year`. Машина раньше 1970 года выпуска маловероятно может быть на ходу с 2015 года, поэтому отфильтровали данные по году выпуска от 1970 по 2025.
- Очищенные таблицы мы конвертировали в csv-файлы.